In [ ]:
import sys

sys.path.insert(0, "..")

import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from pathlib import Path

from dtg.modules.dtg.dtg import DtGModule
from dtg.callbacks.log_animations import ColorizeRainRates
from dtg.data.datamodule import DtGDataModule

In [ ]:
run_name = "dtg_cvpr_dk"
run_dir = Path("..") / "runs" / run_name
ckpt_path = run_dir / "checkpoints" / "best.ckpt"

sources = [
    "rainviewer_4km",
    "wumap_4km",
    "rainviewer_mask_4km",
    "climate_mask_4km",
    "wumap_mask_4km",
]

datamodule = DtGDataModule(
    sources=sources,
    eval_batch_size=1,
    num_workers=0,
)

model = DtGModule.load_from_checkpoint(ckpt_path)
datamodule.set_transforms(model.transforms)
model.cuda().eval()

colorizer = ColorizeRainRates()

In [ ]:
datamodule.setup("test")
dataset = datamodule.test_dataset

In [ ]:
SAMPLE_IDX = 0
BG = "#0e0e0e"
MASKED = "#333333"  # out-of-domain pixels (consistent across all panels)
FIG_W, FIG_H, DPI = 9.6, 9.6, 100  # 960×960 px — both divisible by 16

# ── Colormaps ──────────────────────────────────────────────────────────────
bounds_np = colorizer.bounds.cpu().numpy()
colors_np = colorizer.colors.cpu().numpy() / 255.0
finite_bounds = bounds_np[np.isfinite(bounds_np)]

rain_cmap = ListedColormap(colors_np[: len(finite_bounds)])
rain_cmap.set_bad(MASKED)
rain_norm = BoundaryNorm(
    np.concatenate([[0.0], finite_bounds]), ncolors=len(finite_bounds)
)
RAIN_TICKS = [0.2, 1, 2, 5, 10, 20]
RAIN_LABELS = ["0.2", "1", "2", "5", "10", "20"]

DIFF_VLIM = 5.0  # ± mm/h symmetric range for correction panel
diff_cmap = plt.cm.RdBu_r.copy()  # red = DTG adds rain, blue = DTG removes rain
diff_cmap.set_bad(MASKED)
diff_norm = plt.Normalize(vmin=-DIFF_VLIM, vmax=DIFF_VLIM)
DIFF_TICKS = [-DIFF_VLIM, 0, DIFF_VLIM]
DIFF_LABELS = [f"-{DIFF_VLIM:.0f}", "0", f"+{DIFF_VLIM:.0f}"]

COL_TITLES = [
    "Radar",
    "Weather Stations",
    "DropsToGrid (Ours)",
    "Correction\n(DropsToGrid - Radar)",
]

In [ ]:
# ── Helpers ────────────────────────────────────────────────────────────────
def compute_crop(arr, pad=3):
    """Bounding box of non-NaN pixels with a small border."""
    valid = ~np.isnan(arr)
    rows, cols = np.any(valid, axis=1), np.any(valid, axis=0)
    r0, r1 = np.where(rows)[0][[0, -1]]
    c0, c1 = np.where(cols)[0][[0, -1]]
    h, w = arr.shape
    return np.s_[
        max(0, r0 - pad) : min(h, r1 + pad + 1), max(0, c0 - pad) : min(w, c1 + pad + 1)
    ]


def add_colorbar(fig, cax, cmap, norm, ticks, labels):
    cbar = fig.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), cax=cax)
    cbar.set_ticks(ticks)
    cbar.set_ticklabels(labels)
    cbar.set_label("mm / h", color="white", fontsize=10, labelpad=6)
    cbar.ax.yaxis.set_tick_params(color="white", length=3)
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color="white", fontsize=9)
    cbar.outline.set_edgecolor("#555555")
    cax.set_facecolor(BG)


def render_frame(radar, stations, dtg, correction, title):
    fig = plt.figure(figsize=(FIG_W, FIG_H), dpi=DPI, facecolor=BG)

    # 2×3 grid: [map | map | cbar] per row
    # Row 0: Radar        | Weather Stations | rain colorbar
    # Row 1: DropsToGrid  | Correction       | diff colorbar
    gs = fig.add_gridspec(
        2, 3,
        width_ratios=[1, 1, 0.10],
        left=0.03, right=0.93,
        top=0.82, bottom=0.05,
        hspace=0.20, wspace=0.08,
    )

    ax_radar = fig.add_subplot(gs[0, 0])
    ax_sta   = fig.add_subplot(gs[0, 1])
    cax_rain = fig.add_subplot(gs[0, 2])
    ax_dtg   = fig.add_subplot(gs[1, 0])
    ax_corr  = fig.add_subplot(gs[1, 1])
    cax_diff = fig.add_subplot(gs[1, 2])

    for ax, img, panel_title in zip(
        [ax_radar, ax_sta, ax_dtg, ax_corr],
        [radar, stations, dtg, correction],
        COL_TITLES,
    ):
        cmap = rain_cmap if ax is not ax_corr else diff_cmap
        norm = rain_norm if ax is not ax_corr else diff_norm
        ax.imshow(img, cmap=cmap, norm=norm, interpolation="nearest")
        ax.set_title(panel_title, color="white", fontsize=17, pad=7, fontweight="semibold")
        ax.set_facecolor(BG)
        ax.axis("off")

    add_colorbar(fig, cax_rain, rain_cmap, rain_norm, RAIN_TICKS, RAIN_LABELS)
    add_colorbar(fig, cax_diff, diff_cmap, diff_norm, DIFF_TICKS, DIFF_LABELS)

    fig.suptitle(
        f"DropsToGrid — synthetic demo data\n{title}\n",
        color="white",
        fontsize=25,
        fontweight="bold",
        y=0.97,
    )
    return fig


def sq(t):
    return t.squeeze().numpy()

In [ ]:
with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    sample = dataset[SAMPLE_IDX]
    batch = {k: v.unsqueeze(0).cuda() for k, v in sample.items()}
    out = model.predict_step(batch, 0, return_targets=True)

    radar = out.additional_sources["radar_4km"][0].float().cpu()
    stations = out.context[0, -1:].float().cpu()
    dtg = out.output[0].float().cpu()

    nan_mask = torch.isnan(radar)
    stations = stations.masked_fill(nan_mask, float("nan"))
    dtg = dtg.masked_fill(nan_mask, float("nan"))
    correction = (dtg - radar).masked_fill(nan_mask, float("nan"))

    radar_np = sq(radar)
    crop = compute_crop(radar_np)

    fig = render_frame(
        radar_np[crop],
        sq(stations)[crop],
        sq(dtg)[crop],
        sq(correction)[crop],
        f"Synthetic sample {SAMPLE_IDX}",
    )

output_path = Path("../resources/dtg_fake_demo.png")
fig.savefig(output_path, facecolor=BG, pil_kwargs={"optimize": True})
print(f"Saved frame → {output_path}")